# Prompt Chaining con Flow

Clasificación: **Workflow con Flow.** Mismo chaining secuencial, pero orquestado con la clase `Flow` de CrewAI.

## Por qué usar Flow en vez de llamar a crew.kickoff() directamente

Llamar a `crew.kickoff()` funciona, pero todo queda en una sola operación. Un `Flow` te permite:

1. **Separar pasos con lógica intermedia.** Entre un paso y el siguiente puedes validar inputs, transformar datos o loggear sin ensuciar la crew.
2. **Estado tipado con Pydantic.** En vez de pasar un dict suelto, defines un `BaseModel` con los campos que compartes entre pasos. Si un campo falta o tiene el tipo incorrecto, Pydantic te avisa antes de ejecutar nada.
3. **Composición declarativa.** Cada método es un paso, y los decoradores (`@start`, `@listen`) describen el flujo. Leer el código te dice el orden sin rastrear llamadas.
4. **Base para routing.** Si más adelante necesitas condicionales (ej: país vs ciudad), `@router` se añade al mismo Flow sin reestructurar.

In [1]:
!uv pip install -r requirements.txt --quiet

In [2]:
from dotenv import load_dotenv
import nest_asyncio

load_dotenv()
nest_asyncio.apply()

## Cómo funciona un Flow

Un `Flow` es una clase que hereda de `Flow[MiState]`. El state es un `BaseModel` de Pydantic con los datos que pasan entre pasos.

| Decorador | Qué hace |
|-----------|----------|
| `@start()` | Primer método que se ejecuta al llamar `flow.kickoff()`. |
| `@listen(paso)` | Se ejecuta cuando `paso` termina. Recibe el return value de ese paso. |

El `@start` puede usar `input()` para pedir datos al usuario antes de lanzar la crew. El state queda disponible en todos los pasos siguientes via `self.state`.

```python
class MiFlow(Flow[MiState]):
    @start()
    def pedir_datos(self):
        self.state.tema = input("Tema: ")

    @listen(pedir_datos)
    def ejecutar(self):
        result = MiCrew().crew().kickoff(inputs={"tema": self.state.tema})
        return result.raw
```

In [3]:
from viajes_crew import ViajesCrew

## El Flow de viajes

Tres pasos:
1. `pedir_datos` (start): pide destino, días, personas y presupuesto al usuario via `input()`.
2. `ejecutar_chaining` (listen): lanza la crew con esos datos y devuelve el itinerario.

In [4]:
from pydantic import BaseModel
from crewai.flow.flow import Flow, start, listen


class ViajeState(BaseModel):
    destino: str = ""
    dias: int = 0
    personas: int = 0
    presupuesto: int = 0


class ViajesChainingFlow(Flow[ViajeState]):

    @start()
    def pedir_datos(self):
        print("\n=== Planificador de Viajes ===\n")
        self.state.destino = input("Destino: ")
        self.state.dias = int(input("Días: "))
        self.state.personas = int(input("Personas: "))
        self.state.presupuesto = int(input("Presupuesto (EUR): "))
        print(f"\nPlanificando viaje a {self.state.destino}, {self.state.dias} días, {self.state.personas} personas, {self.state.presupuesto} EUR\n")
        return self.state

    @listen(pedir_datos)
    def ejecutar_chaining(self, state):
        inputs = {
            "destino": state.destino,
            "dias": state.dias,
            "personas": state.personas,
            "presupuesto": state.presupuesto,
        }
        result = ViajesCrew().crew().kickoff(inputs=inputs)
        return result.raw

## Ejecución

Al ejecutar la celda, el notebook te pedirá los datos del viaje en los inputs de abajo.

In [5]:
flow = ViajesChainingFlow()
result = flow.kickoff()
print(result)

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: ViajesChainingFlow                                                                                       │
│  ID: 139afc87-d0e6-42ad-9997-7eeaee6012a8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: ViajesChainingFlow                                                                                       │
│  ID: 139afc87-d0e6-42ad-9997-7eeaee6012a8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Flow started with ID: 139afc87-d0e6-42ad-9997-7eeaee6012a8


=== Planificador de Viajes ===


╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: pedir_datos                                                                                            │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: ViajesCrew                                                                                               │
│  ID: 1f861f68-41b7-49a5-a874-7e7495d98504                                                                       │
│  Final Output: Parece que no tengo acceso directo para ejecutar la consulta de distancias y tiempos reales a    │
│  través de Google Maps API en este momento. Sin embargo, con base en datos típicos y referencias comunes,       │
│  puedo armar el itinerario detallado día a día con horarios aproximados, ordenando las actividades por          │
│  proximidad geográfica y tiempos estándar de desplazamiento en Malta.                                           │
│                                                                                                                 │
│  Haré el desglose con distancias y tiempos estimados basados en experiencias y mapas generales, y costos como   │
│  los indicados en el plan inicial, ajustándolo para máxima coherencia y eficiencia.                             │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  # Itinerario Final para 4 personas en Malta - 10 Días                                                          │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Día 1: Llegada y Exploración Local en Mtarfa                                                                │
│  - **Horario**:                                                                                                 │
│    - 14:00 Llegada al aeropuerto de Malta                                                                       │
│    - 15:00 Traslado a Casa en Mtarfa (aprox. 20 km, 30 min en coche/taxi)                                       │
│    - 16:00 Paseo tranquilo y visita a puntos históricos de Mtarfa (a pie)                                       │
│    - 19:00 Cena en restaurante local - comida típica maltés                                                     │
│  - **Distancias y Tiempos**:                                                                                    │
│    - Aeropuerto a Mtarfa: 20 km, 30 min                                                                         │
│    - Paseo/local: 2 km máximo a pie                                                                             │
│  - **Alojamiento**: Casa en Mtarfa Airbnb                                                                       │
│  - **Coste del día**:                                                                                           │
│    - Comidas (4 pax x 20 EUR) = 80 EUR                                                                          │
│    - Traslado aeropuerto = 30 EUR taxi estimado                                                                 │
│  - **Total día 1**: 110 EUR                                                                                     │
│                                                                                                                 │
│  ---                                                  

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: ejecutar_chaining                                                                                      │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: ViajesChainingFlow                                                                                       │
│  ID: 139afc87-d0e6-42ad-9997-7eeaee6012a8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Trace Batch Finalization ────────────────────────────────────────────╮
│ ✅ Trace batch finalized with session ID: 765c56ff-dfd0-4f81-9108-52aa3217a7a8                                  │
│                                                                                                                 │
│ 🔗 View here: https://app.crewai.com/crewai_plus/trace_batches/765c56ff-dfd0-4f81-9108-52aa3217a7a8             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Parece que no tengo acceso directo para ejecutar la consulta de distancias y tiempos reales a través de Google Maps API en este momento. Sin embargo, con base en datos típicos y referencias comunes, puedo armar el itinerario detallado día a día con horarios aproximados, ordenando las actividades por proximidad geográfica y tiempos estándar de desplazamiento en Malta.

Haré el desglose con distancias y tiempos estimados basados en experiencias y mapas generales, y costos como los indicados en el plan inicial, ajustándolo para máxima coherencia y eficiencia.

---

# Itinerario Final para 4 personas en Malta - 10 Días

---

## Día 1: Llegada y Exploración Local en Mtarfa
- **Horario**: 
  - 14:00 Llegada al aeropuerto de Malta
  - 15:00 Traslado a Casa en Mtarfa (aprox. 20 km, 30 min en coche/taxi)
  - 16:00 Paseo tranquilo y visita a puntos históricos de Mtarfa (a pie)
  - 19:00 Cena en restaurante local - comida típica maltés
- **Distancias y Tiempos**:
  - Aeropuerto a Mtarfa: 20 km, 3